# Metadata Analysis — ADANA CosMx 6K-plex Breast Cancer Cohort

Generate visualisations from patient-level clinical metadata.  
Panels: Demographics, Clinical Tile/Oncoplot, Treatment UpSet, Tumor Burden

## 0. Setup & Configuration

In [1]:
# ── Library imports ──
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(ggplot2)
  library(patchwork)
  library(scales)
  library(forcats)
  library(RColorBrewer)
})

# Check for ComplexUpset (optional)
HAS_COMPLEXUPSET <- tryCatch({
  library(ComplexUpset)
  TRUE
}, error = function(e) FALSE)
cat("ComplexUpset available:", HAS_COMPLEXUPSET, "\n")

ComplexUpset available: FALSE 


In [4]:
# ══════════════════════════════════════════════════════════
# CONFIGURABLE PARAMETERS
# ══════════════════════════════════════════════════════════

# Paths
csv_path   <- "../inputs/sample_metadata_details.csv"
out_dir    <- "../outputs"
study_name <- "METADATA"
full_out_dir <- file.path(out_dir, study_name)

# Receptor source: "path" or "registry"
RECEPTOR_SOURCE <- "path"

# Derive receptor column names
er_col   <- paste0("er_status_",   RECEPTOR_SOURCE)
pr_col   <- paste0("pr_status_",   RECEPTOR_SOURCE)
her2_col <- paste0("her2_status_", RECEPTOR_SOURCE)

# Plot settings
FIG_W      <- 12
FIG_H      <- 8
AXIS_SIZE  <- 7
TITLE_SIZE <- 8
STRIP_SIZE <- 7
DPI        <- 300

cat("Using receptor source:", RECEPTOR_SOURCE, "\n")
cat("ER:", er_col, "| PR:", pr_col, "| HER2:", her2_col, "\n")

Using receptor source: path 
ER: er_status_path | PR: pr_status_path | HER2: her2_status_path 


In [ ]:
# ══════════════════════════════════════════════════════════
# COLOR PALETTES
# ══════════════════════════════════════════════════════════

SUBTYPE_COLORS <- c(
  "Luminal A"      = "#2166AC",
  "Luminal B"      = "#67A9CF",
  "HER2-enriched"  = "#EF8A62",
  "TNBC"           = "#B2182B",
  "Unclassified"   = "#999999"
)

STAGE_COLORS <- c("2" = "#FEE08B", "3" = "#D73027")

GRADE_COLORS <- c(
  "1" = "#A6D96A", "2" = "#FEE08B", "3" = "#F46D43",
  "4" = "#A50026", "Unknown" = "#BDBDBD"
)

RECEPTOR_COLORS <- c(
  "Positive"   = "#D73027",
  "Negative"   = "#4575B4",
  "Borderline" = "#FEE090",
  "Unknown"    = "#BDBDBD"
)

RACE_COLORS <- c(
  "NHB"      = "#1B9E77",
  "HISPANIC" = "#D95F02",
  "API"      = "#7570B3",
  "NHW"      = "#E7298A"
)

MENOPAUSE_COLORS <- c("Pre" = "#91BFDB", "Post" = "#FC8D59")
DIABETES_COLORS  <- c("Yes" = "#E66101", "No" = "#5E3C99")
SMOKING_COLORS   <- c("Ever" = "#E66101", "Never" = "#5E3C99", "Unknown" = "#BDBDBD")
ALCOHOL_COLORS   <- c("Ever" = "#E66101", "Never" = "#5E3C99", "Unknown" = "#BDBDBD")

NA_COLOR <- "grey80"

In [ ]:
# ── Global theme ──
theme_fig1 <- theme_minimal(base_size = 8) +
  theme(
    axis.text      = element_text(size = AXIS_SIZE),
    axis.title     = element_text(size = TITLE_SIZE),
    plot.title     = element_text(size = TITLE_SIZE, face = "bold"),
    strip.text     = element_text(size = STRIP_SIZE),
    legend.text    = element_text(size = AXIS_SIZE),
    legend.title   = element_text(size = TITLE_SIZE),
    panel.grid.minor = element_blank()
  )
theme_set(theme_fig1)

In [ ]:
# ── Create output directories ──
sub_dirs <- c("demographics", "tile_plot", "upset_plot", "tumor_burden", "combined")
for (d in sub_dirs) {
  dir.create(file.path(full_out_dir, d), recursive = TRUE, showWarnings = FALSE)
}
cat("Output directories created under:", full_out_dir, "\n")

In [ ]:
# ── Load and validate CSV ──
meta <- read.csv(csv_path, stringsAsFactors = FALSE)
cat("Dimensions:", nrow(meta), "x", ncol(meta), "\n\n")

# Missingness summary
na_summary <- data.frame(
  column   = names(meta),
  n_miss   = sapply(meta, function(x) sum(is.na(x) | x == "")),
  pct_miss = sapply(meta, function(x) round(100 * mean(is.na(x) | x == ""), 1)),
  stringsAsFactors = FALSE
)
na_summary <- na_summary[order(-na_summary$pct_miss), ]
print(na_summary)

if (any(na_summary$pct_miss > 5)) {
  warning("Columns with >5% missing: ",
          paste(na_summary$column[na_summary$pct_miss > 5], collapse = ", "))
}

In [ ]:
# ══════════════════════════════════════════════════════════
# DATA CLEANING
# ══════════════════════════════════════════════════════════

# Fix truncated values
meta$smoking_status_en[meta$smoking_status_en == "Unkno"] <- "Unknown"

# Fix HER2 "Borderli" → "Borderline"
for (col in grep("her2_status", names(meta), value = TRUE)) {
  meta[[col]][meta[[col]] == "Borderli"] <- "Borderline"
}

# Clean grade: keep numeric values, mark rest as "Unknown"
meta$grade_clean <- ifelse(meta$grade %in% c("1", "2", "3", "4"),
                           as.character(meta$grade), "Unknown")
meta$grade_num <- suppressWarnings(as.integer(meta$grade_clean))

# Stage as factor
meta$stage <- factor(meta$stage)

# Recode binary columns to labels
meta$menopausal_label <- ifelse(meta$menopausal_status == 1, "Post", "Pre")
meta$diabetes_label   <- ifelse(meta$diabetes_status == 1, "Yes", "No")

# Standardize receptor columns
recode_receptor <- function(x) {
  x <- trimws(x)
  dplyr::case_when(
    x %in% c("Positive", "Pos") ~ "Positive",
    x %in% c("Negative", "Neg") ~ "Negative",
    x == "Borderline"            ~ "Borderline",
    TRUE                          ~ "Unknown"
  )
}
meta[[er_col]]   <- recode_receptor(meta[[er_col]])
meta[[pr_col]]   <- recode_receptor(meta[[pr_col]])
meta[[her2_col]] <- recode_receptor(meta[[her2_col]])

cat("Cleaning complete. Verifying recoded values:\n")
cat("\nER:", table(meta[[er_col]]), "\n")
cat("PR:", table(meta[[pr_col]]), "\n")
cat("HER2:", table(meta[[her2_col]]), "\n")
cat("Grade:", table(meta$grade_clean), "\n")
cat("Smoking:", table(meta$smoking_status_en), "\n")

## 1. Molecular Subtyping

In [ ]:
# ══════════════════════════════════════════════════════════
# MOLECULAR SUBTYPING
# ══════════════════════════════════════════════════════════
# Logic:
#   TNBC:           ER-, PR-, HER2-
#   HER2-enriched:  HER2+, ER-, PR-
#   Luminal B:      (ER+ or PR+) AND (HER2+ OR grade >= 3)
#   Luminal A:      (ER+ or PR+), HER2-, grade <= 2 or Unknown
#   Unclassified:   anything else (e.g., missing receptor status)
#
# Note: "Borderline" HER2 is treated as NOT positive for subtyping.
# Patients with Unknown grade who are ER/PR+ and HER2- are assigned
# Luminal A (clinically most likely).

meta <- meta %>%
  mutate(
    er_pos   = .data[[er_col]]   == "Positive",
    pr_pos   = .data[[pr_col]]   == "Positive",
    her2_pos = .data[[her2_col]] == "Positive",
    molecular_subtype = case_when(
      !er_pos & !pr_pos & !her2_pos                                        ~ "TNBC",
      her2_pos & !er_pos & !pr_pos                                         ~ "HER2-enriched",
      (er_pos | pr_pos) & (her2_pos | (!is.na(grade_num) & grade_num >= 3)) ~ "Luminal B",
      (er_pos | pr_pos) & !her2_pos & (is.na(grade_num) | grade_num <= 2)  ~ "Luminal A",
      TRUE                                                                  ~ "Unclassified"
    ),
    molecular_subtype = factor(molecular_subtype,
      levels = c("Luminal A", "Luminal B", "HER2-enriched", "TNBC", "Unclassified"))
  )

cat("Molecular subtype distribution:\n")
print(table(meta$molecular_subtype, useNA = "ifany"))
cat("\nPercentages:\n")
print(round(prop.table(table(meta$molecular_subtype)) * 100, 1))

## 2. Panel B — Demographics (Age, BMI, Race/Ethnicity)

In [ ]:
# ── B1: Age at diagnosis — density by subtype ──
age_medians <- meta %>%
  group_by(molecular_subtype) %>%
  summarise(median_age = median(age_at_dx, na.rm = TRUE), .groups = "drop")

p_b1 <- ggplot(meta, aes(x = age_at_dx, fill = molecular_subtype, color = molecular_subtype)) +
  geom_density(alpha = 0.3, linewidth = 0.5) +
  geom_rug(alpha = 0.3, length = unit(0.03, "npc")) +
  geom_vline(data = age_medians,
             aes(xintercept = median_age, color = molecular_subtype),
             linetype = "dashed", linewidth = 0.5, show.legend = FALSE) +
  scale_fill_manual(values = SUBTYPE_COLORS) +
  scale_color_manual(values = SUBTYPE_COLORS) +
  labs(x = "Age at Diagnosis", y = "Density", fill = "Subtype", color = "Subtype") +
  theme(legend.position = "bottom",
        legend.key.size = unit(0.3, "cm"))
p_b1

In [ ]:
# ── B2: BMI — violin by subtype ──
p_b2 <- ggplot(meta, aes(x = molecular_subtype, y = bmi, fill = molecular_subtype)) +
  geom_violin(alpha = 0.5, scale = "width") +
  geom_boxplot(width = 0.15, outlier.size = 0.5, fill = "white", alpha = 0.7) +
  geom_hline(yintercept = 30, linetype = "dashed", color = "red", linewidth = 0.4) +
  annotate("text", x = 0.6, y = 31, label = "Obesity threshold (BMI 30)",
           hjust = 0, size = 2, color = "red") +
  scale_fill_manual(values = SUBTYPE_COLORS) +
  labs(x = NULL, y = "BMI") +
  theme(legend.position = "none",
        axis.text.x = element_text(angle = 30, hjust = 1))
p_b2

In [ ]:
# ── B3: Race/ethnicity — stacked horizontal bars ──
p_b3 <- meta %>%
  count(molecular_subtype, race_ethnicity) %>%
  group_by(molecular_subtype) %>%
  mutate(pct = n / sum(n)) %>%
  ungroup() %>%
  ggplot(aes(x = molecular_subtype, y = pct, fill = race_ethnicity)) +
  geom_col(position = "fill", width = 0.7) +
  coord_flip() +
  scale_fill_manual(values = RACE_COLORS) +
  scale_y_continuous(labels = percent_format()) +
  labs(x = NULL, y = "Proportion", fill = "Race/Ethnicity") +
  theme(legend.position = "bottom",
        legend.key.size = unit(0.3, "cm"),
        panel.grid.major.y = element_blank())
p_b3

In [ ]:
# ── Compose Panel B ──
panel_b <- p_b1 / (p_b2 | p_b3) +
  plot_annotation(tag_levels = list(c("B1", "B2", "B3")))

ggsave(file.path(full_out_dir, "demographics", "panel_B.pdf"),
       panel_b, width = FIG_W, height = FIG_H)
cat("Saved Panel B\n")
panel_b

## 3. Panel C — Clinical Tile Plot (Oncoplot-style)

**Approach:** Single unified fill factor (`variable::value` keys) to avoid `ggnewscale` complexity.  
Each row's values are disjoint, so one master palette covers all rows without collisions.

In [ ]:
# ── Prepare tile data ──

# Sort patients by subtype → stage → grade
meta <- meta %>%
  arrange(molecular_subtype, stage, grade_clean) %>%
  mutate(patient_idx = row_number())

# Variables to display (using cleaned/labeled versions)
tile_var_map <- c(
  "ER"          = er_col,
  "PR"          = pr_col,
  "HER2"        = her2_col,
  "Stage"       = "stage",
  "Grade"       = "grade_clean",
  "Menopausal"  = "menopausal_label",
  "Diabetes"    = "diabetes_label",
  "Smoking"     = "smoking_status_en",
  "Alcohol"     = "alcohol_status_en"
)

# Build long-format data
tile_list <- lapply(seq_along(tile_var_map), function(i) {
  display_name <- names(tile_var_map)[i]
  col_name     <- tile_var_map[i]
  data.frame(
    patient_idx       = meta$patient_idx,
    molecular_subtype = meta$molecular_subtype,
    variable          = display_name,
    value             = as.character(meta[[col_name]]),
    stringsAsFactors  = FALSE
  )
})
tile_df <- do.call(rbind, tile_list)

# Order variables (top to bottom)
var_order <- rev(names(tile_var_map))
tile_df$variable <- factor(tile_df$variable, levels = var_order)

# Create unique fill key per variable::value
tile_df <- tile_df %>%
  mutate(value = ifelse(is.na(value) | value == "", NA_character_, value),
         fill_key = ifelse(is.na(value), "NA",
                           paste0(as.character(variable), "::", value)))

cat("Tile data:", nrow(tile_df), "rows\n")
cat("Unique fill keys:", length(unique(tile_df$fill_key)), "\n")

In [ ]:
# ── Build master palette ──

# Per-variable color lookup
var_palettes <- list(
  ER         = RECEPTOR_COLORS,
  PR         = RECEPTOR_COLORS,
  HER2       = RECEPTOR_COLORS,
  Stage      = STAGE_COLORS,
  Grade      = GRADE_COLORS,
  Menopausal = MENOPAUSE_COLORS,
  Diabetes   = DIABETES_COLORS,
  Smoking    = SMOKING_COLORS,
  Alcohol    = ALCOHOL_COLORS
)

all_keys <- unique(tile_df$fill_key[!is.na(tile_df$fill_key) & tile_df$fill_key != "NA"])

master_palette <- sapply(all_keys, function(k) {
  parts <- strsplit(k, "::")[[1]]
  var_name <- parts[1]
  val      <- parts[2]
  pal <- var_palettes[[var_name]]
  if (!is.null(pal) && val %in% names(pal)) {
    return(pal[val])
  }
  return("#BDBDBD")  # fallback for unexpected values
})
master_palette["NA"] <- NA_COLOR

cat("Master palette entries:", length(master_palette), "\n")
print(master_palette)

In [ ]:
# ── Plot tile ──

# Subtype color bar (top annotation)
p_subtype_bar <- ggplot(meta, aes(x = patient_idx, y = 1, fill = molecular_subtype)) +
  geom_tile() +
  scale_fill_manual(values = SUBTYPE_COLORS, name = "Subtype") +
  scale_x_continuous(expand = c(0, 0)) +
  theme_void() +
  theme(legend.position = "right",
        legend.text = element_text(size = 6),
        legend.title = element_text(size = 7, face = "bold"),
        legend.key.size = unit(0.3, "cm")) +
  labs(y = NULL)

# Main tile plot
p_tile <- ggplot(tile_df, aes(x = patient_idx, y = variable, fill = fill_key)) +
  geom_tile(color = NA) +
  scale_fill_manual(values = master_palette, na.value = NA_COLOR, guide = "none") +
  scale_x_continuous(expand = c(0, 0)) +
  labs(x = "Patients (sorted by subtype / stage / grade)", y = NULL) +
  theme(
    axis.text.x  = element_blank(),
    axis.ticks.x = element_blank(),
    panel.grid   = element_blank()
  )

# Compose
panel_c_main <- p_subtype_bar / p_tile +
  plot_layout(heights = c(1, 10))
panel_c_main

In [ ]:
# ── Tile plot legends ──
# Build a legend panel with small key plots for each variable group

make_legend <- function(values, colors, title) {
  df <- data.frame(x = 1, y = seq_along(values),
                   val = factor(values, levels = values))
  ggplot(df, aes(x, y, fill = val)) +
    geom_tile(width = 0.5, height = 0.8) +
    geom_text(aes(label = val), size = 2, hjust = -0.3) +
    scale_fill_manual(values = colors, guide = "none") +
    scale_x_continuous(limits = c(0.5, 3)) +
    labs(title = title) +
    theme_void() +
    theme(plot.title = element_text(size = 6, face = "bold"))
}

leg_receptor <- make_legend(
  c("Positive", "Negative", "Borderline", "Unknown"),
  RECEPTOR_COLORS, "ER/PR/HER2")
leg_stage <- make_legend(names(STAGE_COLORS), STAGE_COLORS, "Stage")
leg_grade <- make_legend(names(GRADE_COLORS), GRADE_COLORS, "Grade")
leg_binary <- make_legend(
  c("Yes/Post/Ever", "No/Pre/Never", "Unknown"),
  c("Yes/Post/Ever" = "#E66101", "No/Pre/Never" = "#5E3C99", "Unknown" = "#BDBDBD"),
  "Binary Variables")

legend_panel <- (leg_receptor | leg_stage | leg_grade | leg_binary) +
  plot_layout(widths = c(1, 1, 1, 1))

# Full Panel C with legends
panel_c <- panel_c_main / legend_panel +
  plot_layout(heights = c(8, 2))

ggsave(file.path(full_out_dir, "tile_plot", "panel_C.pdf"),
       panel_c, width = 14, height = 7)
cat("Saved Panel C\n")
panel_c

## 4. Panel D — Treatment UpSet Plot

In [ ]:
# ── Prepare treatment data ──
tx_cols_orig <- c("received_chemo", "received_rad", "received_horm", "received_surgery")
tx_labels    <- c("Chemotherapy", "Radiation", "Hormone Therapy", "Surgery")

tx_df <- meta %>%
  select(patient_idx, molecular_subtype, all_of(tx_cols_orig))

# Rename to display labels
names(tx_df)[match(tx_cols_orig, names(tx_df))] <- tx_labels

# Convert 0/1 to logical
tx_df <- tx_df %>%
  mutate(across(all_of(tx_labels), ~ as.logical(.)))

cat("Treatment frequencies:\n")
for (tx in tx_labels) cat(tx, ":", sum(tx_df[[tx]]), "/", nrow(tx_df), "\n")

In [ ]:
# ── UpSet Plot ──

if (HAS_COMPLEXUPSET) {
  panel_d <- upset(
    tx_df, tx_labels,
    base_annotations = list(
      "Intersection size" = intersection_size(
        mapping = aes(fill = molecular_subtype)
      ) + scale_fill_manual(values = SUBTYPE_COLORS)
    ),
    min_size = 5,
    width_ratio = 0.3,
    set_sizes = upset_set_size() +
      theme(axis.text.x = element_text(angle = 45, hjust = 1))
  )
} else {
  # ── Manual UpSet fallback ──
  # Compute combination strings
  tx_df$combo <- apply(tx_df[tx_labels], 1, function(r) {
    active <- tx_labels[r]
    if (length(active) == 0) return("None")
    paste(active, collapse = " + ")
  })

  # Count per combo x subtype
  combo_counts <- tx_df %>%
    count(combo, molecular_subtype) %>%
    group_by(combo) %>%
    mutate(total = sum(n)) %>%
    ungroup() %>%
    filter(total >= 5)

  combo_order <- combo_counts %>%
    distinct(combo, total) %>%
    arrange(desc(total)) %>%
    pull(combo)

  combo_counts$combo <- factor(combo_counts$combo, levels = rev(combo_order))

  # Bar plot
  p_bars <- ggplot(combo_counts, aes(x = combo, y = n, fill = molecular_subtype)) +
    geom_col() +
    scale_fill_manual(values = SUBTYPE_COLORS) +
    coord_flip() +
    labs(y = "Number of Patients", x = NULL, fill = "Subtype") +
    theme(legend.position = "bottom",
          legend.key.size = unit(0.3, "cm"),
          panel.grid.major.y = element_blank())

  # Dot matrix
  dot_data <- expand.grid(
    combo     = combo_order,
    treatment = tx_labels,
    stringsAsFactors = FALSE
  ) %>%
    rowwise() %>%
    mutate(active = grepl(treatment, combo, fixed = TRUE)) %>%
    ungroup() %>%
    mutate(combo = factor(combo, levels = rev(combo_order)))

  # Segments connecting active dots within each combo
  seg_data <- dot_data %>%
    filter(active) %>%
    group_by(combo) %>%
    summarise(ymin = min(as.integer(factor(treatment, levels = tx_labels))),
              ymax = max(as.integer(factor(treatment, levels = tx_labels))),
              .groups = "drop") %>%
    filter(ymin != ymax)

  dot_data$treatment_num <- as.integer(factor(dot_data$treatment, levels = tx_labels))

  p_dots <- ggplot(dot_data, aes(x = combo, y = treatment_num)) +
    geom_segment(data = seg_data,
                 aes(x = combo, xend = combo, y = ymin, yend = ymax),
                 linewidth = 0.5, color = "grey30") +
    geom_point(aes(color = active), size = 2) +
    scale_color_manual(values = c("TRUE" = "#333333", "FALSE" = "grey85"), guide = "none") +
    scale_y_continuous(breaks = seq_along(tx_labels), labels = tx_labels) +
    coord_flip() +
    labs(x = NULL, y = NULL) +
    theme(panel.grid = element_blank(),
          axis.text.y = element_blank(),
          axis.ticks.y = element_blank())

  panel_d <- (p_bars | p_dots) + plot_layout(widths = c(3, 1))
}

ggsave(file.path(full_out_dir, "upset_plot", "panel_D.pdf"),
       panel_d, width = 10, height = 6)
cat("Saved Panel D\n")
panel_d

## 5. Panel E — Tumor Burden

In [ ]:
# ── E1: Tumor size by subtype ──
p_e1 <- ggplot(meta %>% filter(!is.na(tumor_size_mm)),
               aes(x = molecular_subtype, y = tumor_size_mm, fill = molecular_subtype)) +
  geom_violin(alpha = 0.5, scale = "width") +
  geom_boxplot(width = 0.15, outlier.size = 0.5, fill = "white", alpha = 0.7) +
  geom_hline(yintercept = 20, linetype = "dashed", color = "#E69F00", linewidth = 0.4) +
  geom_hline(yintercept = 50, linetype = "dashed", color = "#D55E00", linewidth = 0.4) +
  annotate("text", x = 5.3, y = 21, label = "T1/T2 (20 mm)",
           hjust = 0, size = 2, color = "#E69F00") +
  annotate("text", x = 5.3, y = 51, label = "T2/T3 (50 mm)",
           hjust = 0, size = 2, color = "#D55E00") +
  scale_fill_manual(values = SUBTYPE_COLORS) +
  coord_cartesian(clip = "off") +
  labs(x = NULL, y = "Tumor Size (mm)") +
  theme(legend.position = "none",
        axis.text.x = element_text(angle = 30, hjust = 1),
        plot.margin = margin(5, 30, 5, 5))
p_e1

In [ ]:
# ── E2: Lymph node positivity by subtype ──
ln_summary <- meta %>%
  filter(!is.na(reg_nodes_pos)) %>%
  mutate(ln_positive = reg_nodes_pos > 0) %>%
  group_by(molecular_subtype) %>%
  summarise(
    n_total = n(),
    n_pos   = sum(ln_positive),
    pct_pos = mean(ln_positive),
    .groups = "drop"
  )

p_e2 <- ggplot(ln_summary, aes(x = molecular_subtype, y = pct_pos, fill = molecular_subtype)) +
  geom_col(width = 0.7, alpha = 0.8) +
  geom_text(aes(label = paste0(n_pos, "/", n_total)),
            vjust = -0.3, size = 2.5) +
  scale_fill_manual(values = SUBTYPE_COLORS) +
  scale_y_continuous(labels = percent_format(), expand = expansion(mult = c(0, 0.15))) +
  labs(x = NULL, y = "Proportion LN Positive") +
  theme(legend.position = "none",
        axis.text.x = element_text(angle = 30, hjust = 1),
        panel.grid.major.x = element_blank())
p_e2

In [ ]:
# ── Compose Panel E ──
panel_e <- (p_e1 | p_e2) +
  plot_annotation(tag_levels = list(c("E1", "E2")))

ggsave(file.path(full_out_dir, "tumor_burden", "panel_E.pdf"),
       panel_e, width = 10, height = 5)
cat("Saved Panel E\n")
panel_e

## 6. Combined PDF Report

In [ ]:
# ── Single PDF with all panels (one per page) ──
pdf_path <- file.path(full_out_dir, "combined", "Figure1_all_panels.pdf")
pdf(pdf_path, width = FIG_W, height = FIG_H)

print(panel_b)
print(panel_c)
print(panel_d)
print(panel_e)

dev.off()
cat("Combined PDF saved to:", pdf_path, "\n")

## 7. Summary Statistics

In [ ]:
# ── Key stats per subtype (for manuscript text) ──
summary_stats <- meta %>%
  group_by(molecular_subtype) %>%
  summarise(
    N           = n(),
    Pct         = round(n() / nrow(meta) * 100, 1),
    Median_Age  = median(age_at_dx, na.rm = TRUE),
    Mean_BMI    = round(mean(bmi, na.rm = TRUE), 1),
    Median_Tumor_mm = median(tumor_size_mm, na.rm = TRUE),
    Pct_LN_Pos  = round(mean(reg_nodes_pos > 0, na.rm = TRUE) * 100, 1),
    Pct_Chemo   = round(mean(received_chemo == 1, na.rm = TRUE) * 100, 1),
    Pct_Rad     = round(mean(received_rad == 1, na.rm = TRUE) * 100, 1),
    Pct_Horm    = round(mean(received_horm == 1, na.rm = TRUE) * 100, 1),
    .groups = "drop"
  )

print(summary_stats)

# Overall cohort summary
cat("\n── Overall Cohort ──\n")
cat("N =", nrow(meta), "\n")
cat("Median age:", median(meta$age_at_dx, na.rm = TRUE), "\n")
cat("Mean BMI:", round(mean(meta$bmi, na.rm = TRUE), 1), "\n")
cat("Race/Ethnicity:\n")
print(table(meta$race_ethnicity))
cat("\nStage:\n")
print(table(meta$stage))

In [ ]:
sessionInfo()